# 03 Features

Hand-crafted features for the logistic regression baseline:

0. Split each domain into subdomain / name / suffix (tldextract)
1. Character features and 2-/3-gram frequencies
2. Markov chain score
3. Vowel features
4. TLD features

## Step 0: Split domains with tldextract

We split each domain into **subdomain**, **name** and **suffix** using `tldextract`
(not a simple split at the last dot, which gets `bbc.co.uk` wrong). We use the suffix
list bundled with tldextract 5.4.0, plus its private-suffix section (dynamic-DNS
services such as `ddns.net`) and 7 extra endings used by DGA families, so everyone
gets the same split.

All character features are computed on **text** = subdomain + name without dots,
and the TLD features on **suffix**, so the with/without-TLD experiments stay clean.

First, a quick check on a few example domains, including tricky ones.

In [1]:
# Let the notebook import our code from src/ (the project root is one folder up)
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.features.split import split_domain

# Expected: bbc.co.uk -> name 'bbc', suffix 'co.uk'
for d in ["google.com", "mail.google.com", "bbc.co.uk", "kneemanualgirl.com", "abc.ddns.net"]:
    print(d, "->", split_domain(d))

google.com -> ('', 'google', 'com')
mail.google.com -> ('mail', 'google', 'com')
bbc.co.uk -> ('', 'bbc', 'co.uk')
kneemanualgirl.com -> ('', 'kneemanualgirl', 'com')
abc.ddns.net -> ('', 'abc', 'ddns.net')


### Split the full dataset

We load the same 80/10/10 train/validation/test split as the baseline
(`load_splits()` from `src/data/dataset.py`) and add the **subdomain**, **name**,
**suffix** and **text** columns to each part.

In [2]:
from src.data.dataset import load_splits
from src.features.split import add_split_columns

# Same split as the baseline notebook, so all models see the same rows
train, val, test = load_splits()

# Add subdomain / name / suffix / text to each part (takes about a minute)
train = add_split_columns(train)
val = add_split_columns(val)
test = add_split_columns(test)

print(f"train: {len(train):,}  val: {len(val):,}  test: {len(test):,}")
print(train[["domain", "label", "family", "subdomain", "name", "suffix", "text"]].head(10))

# Sanity checks on all rows
for part_name, part in [("train", train), ("val", val), ("test", test)]:
    print(part_name,
          "| empty names:", (part["name"] == "").sum(),
          "| empty suffixes:", (part["suffix"] == "").sum())

train: 1,599,486  val: 199,936  test: 199,913
                           domain  label     family subdomain  \
0  6j29qz11hsefgh7oqsr1vpapri.com    dga   gameover             
1     qckuygxeyemxgongvylveeoh.cx    dga     necurs             
2                   arenaplus.net  legit      legit             
3                  fd5q27kyqb.com  legit      legit             
4              eksisozluk1923.com  legit      legit             
5                      ucngfq.com    dga      virut             
6                 mincetur.gob.pe  legit      legit             
7                      363ce3.net    dga  tinyfluff             
8                  w5pti45uow.net    dga    urlzone             
9                         acmb.cd    dga    qsnatch             

                         name  suffix                        text  
0  6j29qz11hsefgh7oqsr1vpapri     com  6j29qz11hsefgh7oqsr1vpapri  
1    qckuygxeyemxgongvylveeoh      cx    qckuygxeyemxgongvylveeoh  
2                   arenaplus     

### Check: every domain has a name and a suffix

A domain with an empty name or suffix means tldextract could not split it, which
would distort every feature. The cell below counts these rows, shows that the
previously problematic domains are now split correctly, and stops with an error if
any remain.

In [3]:
# 1) Count empty names / suffixes in every part (should all be 0 after the fix)
for part_name, part in [("train", train), ("val", val), ("test", test)]:
    print(part_name,
          "| empty names:", (part["name"] == "").sum(),
          "| empty suffixes:", (part["suffix"] == "").sum())

# 2) Show that the problem cases are now split correctly
examples = train[train["suffix"].isin(["bazar", "bazaar", "bit"]) | train["domain"].isin(["gov.bw", "ac.pk"])]
print(examples[["domain", "family", "subdomain", "name", "suffix"]].head(8))

# 3) Stop with an error if the problem ever comes back
for part_name, part in [("train", train), ("val", val), ("test", test)]:
    assert (part["name"] == "").sum() == 0, f"{part_name}: empty names"
    assert (part["suffix"] == "").sum() == 0, f"{part_name}: empty suffixes"
print("\nSplit check passed: no empty names or suffixes.")

train | empty names: 0 | empty suffixes: 0
val | empty names: 0 | empty suffixes: 0
test | empty names: 0 | empty suffixes: 0
                  domain         family subdomain          name  suffix
27        vuywibuf.bazar  bazarbackdoor                vuywibuf   bazar
59        aqewufuf.bazar  bazarbackdoor                aqewufuf   bazar
187       sonetoag.bazar  bazarbackdoor                sonetoag   bazar
223       ydidsoli.bazar  bazarbackdoor                ydidsoli   bazar
439       owywibty.bazar  bazarbackdoor                owywibty   bazar
487       omfeudag.bazar  bazarbackdoor                omfeudag   bazar
529       ucidizli.bazar  bazarbackdoor                ucidizli   bazar
636  eadhimecfhit.bazaar  bazarbackdoor            eadhimecfhit  bazaar

Split check passed: no empty names or suffixes.


**Finding.** At first, 23,743 train domains (all DGA) had an empty suffix: they use
endings from alternative DNS systems that tldextract does not know: `.bazar` /
`.bazaar` (bazarbackdoor), `.bit` (necurs), `.load` (gozi), `.oss` / `.geek` /
`.web` (other_dga). tldextract then put the ending in `name` (e.g. `edwiudli.bazar`
→ name `bazar`). Also 8 legit domains are themselves official suffixes (e.g.
`gov.bw`) and had an empty name.

**Fix.** These 7 endings are passed to tldextract (`EXTRA_SUFFIXES` in
`src/features/split.py`); for the 8 `gov.bw`-style domains, the first label of the
suffix becomes the name (to confirm with the team).

**Note.** These endings appear only in DGA domains, so the TLD alone identifies
these families: one reason we test every model with and without the TLD.

### Check: domains with a subdomain

Domains that still have a subdomain after the split, with their `text` column.

In [4]:
print(train[train["subdomain"] != ""][["domain", "subdomain", "name", "text"]].head(10))

                                                 domain  \
291              e9e10c8bb9cb2465686d0f825a25dadc.co.cc   
1255             3c52819a24e6df2c565c9e1cb5ff5233.cz.cc   
1550                                 ns1.backdates4.net   
1696                          www1.runprettyquarter.art   
1728  oeldhchgibwccxchdeebchdeecegqfucegqfa.brazilso...   
1903                      ffeearwedeglhpwh.mynumber.org   
2067                                       slbla.uy.com   
2163  m0d1854739a115ca987292a04214be75d955c3.brazils...   
2875                            cditpjxmgdy.dynserv.com   
3498             163b628e3ae5f90d3804ea83d96eaa1c.co.cc   

                                              subdomain              name  \
291                    e9e10c8bb9cb2465686d0f825a25dadc                co   
1255                   3c52819a24e6df2c565c9e1cb5ff5233                cz   
1550                                                ns1        backdates4   
1696                                      

**Dynamic DNS.** Many DGA domains are created under free services (`ddns.net`,
`duckdns.org`, ...). With `include_psl_private_domains=True`, tldextract treats these
as suffixes, so the random part becomes the name. Services missing from that list
(`co.cc`, `dynserv.com`, ...) still have their random part in `text`
(subdomain + name, no dots), which is what all character features use.